<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Ch7_Transactions_Concurrency_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7: Transactions and Concurrency Control

**Domain:** ระบบธนาคาร / โอนเงิน (Banking & Money Transfer)

ใน Lab นี้เราจะใช้ PostgreSQL ผ่าน **Neon** เพื่อจำลองสถานการณ์ที่มีหลาย Transaction เข้าถึงข้อมูลบัญชีธนาคารพร้อมกัน แล้วสังเกตปัญหาที่เกิดขึ้นจริง พร้อมวิธีแก้ไข

**สิ่งที่จะได้ทำ**
- **Lab A** — สังเกตพฤติกรรมของ Isolation Level ต่างๆ (Dirty Read, Non-repeatable Read)
- **Lab B** — จำลองปัญหา Lost Update แล้วแก้ไขด้วย `SELECT ... FOR UPDATE`
- **Lab C** — จำลอง Deadlock ระหว่างสอง Transaction และอ่าน error message

> **หมายเหตุ:** ใน Lab นี้เราจะเปิด 2 การเชื่อมต่อ (connection) พร้อมกัน แทนตัวแทนของ **Transaction 1 (T1)** และ **Transaction 2 (T2)** ที่ทำงาน "พร้อมกัน" — ต้องรันเซลล์ตามลำดับที่กำหนดไว้(ห้ามข้ามลำดับ) เพื่อให้เห็นการ interleave ของสอง transaction


## 0. Setup — ติดตั้งและเชื่อมต่อฐานข้อมูล

In [1]:
!pip install psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 24.9 MB/s eta 0:00:00


เก็บ connection string ของ Neon ไว้ใน **Colab Secrets** ด้วยชื่อ `NEON_CONNECTION_STRING`
(เมนู 🔑 ทางซ้ายของ Colab → Add new secret) เหมือนที่เคยทำใน Chapter 6

In [2]:
import psycopg2
from google.colab import userdata

CONN_STRING = userdata.get('NEON_CONNECTION_STRING')

def get_conn(autocommit=False):
    """เปิด connection ใหม่หนึ่งเส้น — ใช้แทน 'session' ของแต่ละ transaction"""
    conn = psycopg2.connect(CONN_STRING)
    conn.autocommit = autocommit
    return conn

def run(conn, sql, params=None, fetch=True):
    """รันคำสั่ง SQL บน connection ที่ระบุ และคืนผลลัพธ์เป็น list (ถ้ามี)"""
    cur = conn.cursor()
    cur.execute(sql, params)
    if fetch and cur.description:
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
        cur.close()
        return cols, rows
    cur.close()
    return None, None

def show(cols, rows):
    if cols is None:
        print("(no result set)")
        return
    print(" | ".join(cols))
    for r in rows:
        print(" | ".join(str(v) for v in r))

print("เชื่อมต่อ Neon สำเร็จ พร้อมใช้งาน get_conn() และ run()")

เชื่อมต่อ Neon สำเร็จ พร้อมใช้งาน get_conn() และ run()


## 1. เตรียมข้อมูลตั้งต้น (Seed Data)

สร้างตาราง `accounts` จำลองบัญชีธนาคาร 4 บัญชี

In [3]:
setup_conn = get_conn(autocommit=True)
run(setup_conn, """
DROP TABLE IF EXISTS accounts;
CREATE TABLE accounts (
    account_id   INT PRIMARY KEY,
    owner_name   TEXT NOT NULL,
    balance      NUMERIC(12,2) NOT NULL CHECK (balance >= 0)
);
""", fetch=False)

run(setup_conn, """
INSERT INTO accounts (account_id, owner_name, balance) VALUES
    (1, 'Alice',   5000.00),
    (2, 'Bob',     3000.00),
    (3, 'Charlie', 10000.00),
    (4, 'Diana',   2000.00);
""", fetch=False)

cols, rows = run(setup_conn, "SELECT * FROM accounts ORDER BY account_id;")
show(cols, rows)
setup_conn.close()

account_id | owner_name | balance
1 | Alice | 5000.00
2 | Bob | 3000.00
3 | Charlie | 10000.00
4 | Diana | 2000.00


---
## Lab A — Isolation Levels

### A1. Dirty Read เกิดขึ้นจริงหรือไม่ใน PostgreSQL?

ทฤษฎีบอกว่า `READ UNCOMMITTED` อาจทำให้เกิด Dirty Read (อ่านข้อมูลที่ยังไม่ commit) แต่ PostgreSQL ปฏิบัติต่อ `READ UNCOMMITTED` เหมือนกับ `READ COMMITTED` เสมอ — Lab นี้จะพิสูจน์ว่าจริงหรือไม่

**ลำดับขั้นตอน:** T1 แก้ไขยอดเงินแต่ยังไม่ commit → T2 อ่านค่าที่ระดับ READ UNCOMMITTED → สังเกตว่า T2 เห็นค่าเก่าหรือค่าใหม่

**ขั้นที่ 1 — T1: เริ่ม transaction และหักเงิน Alice (ยังไม่ commit)**

In [4]:
conn_t1 = get_conn()
run(conn_t1, "BEGIN;", fetch=False)
run(conn_t1, "UPDATE accounts SET balance = balance - 1000 WHERE account_id = 1;", fetch=False)
print("T1: หักเงิน Alice ไป 1000 แล้ว (ยังไม่ COMMIT)")

T1: หักเงิน Alice ไป 1000 แล้ว (ยังไม่ COMMIT)


**ขั้นที่ 2 — T2: เปิด transaction ระดับ READ UNCOMMITTED แล้วอ่านยอดเงิน Alice**

In [5]:
conn_t2 = get_conn()
run(conn_t2, "BEGIN ISOLATION LEVEL READ UNCOMMITTED;", fetch=False)
cols, rows = run(conn_t2, "SELECT balance FROM accounts WHERE account_id = 1;")
show(cols, rows)
run(conn_t2, "COMMIT;", fetch=False)

balance
5000.00


(None, None)

**ขั้นที่ 3 — T1: ROLLBACK (ยกเลิกการหักเงิน เพื่อให้ข้อมูลกลับมาเหมือนเดิมก่อนทำ Lab ถัดไป)**

In [6]:
run(conn_t1, "ROLLBACK;", fetch=False)
conn_t1.close()
print("T1: ROLLBACK แล้ว — ยอดเงิน Alice กลับเป็น 5000 เหมือนเดิม")

T1: ROLLBACK แล้ว — ยอดเงิน Alice กลับเป็น 5000 เหมือนเดิม


**PostgreSQL** ไม่มี Dirty Read เพราะสถาปัตยกรรม MVCC (Multi-Version Concurrency Control)

### A2. Non-repeatable Read — READ COMMITTED vs REPEATABLE READ

**ลำดับขั้นตอน:** T1 อ่านยอดเงิน Bob 2 ครั้ง โดยมี T2 มา UPDATE และ COMMIT คั่นกลาง — ทดสอบที่ READ COMMITTED ก่อน แล้วเทียบกับ REPEATABLE READ

**ขั้นที่ 1 — T1: เปิด transaction ระดับ READ COMMITTED แล้วอ่านยอดเงิน Bob ครั้งที่ 1**

In [7]:
conn_t1 = get_conn()
run(conn_t1, "BEGIN ISOLATION LEVEL READ COMMITTED;", fetch=False)
cols, rows = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 2;")
print("T1 อ่านครั้งที่ 1:")
show(cols, rows)

T1 อ่านครั้งที่ 1:
balance
3000.00


**ขั้นที่ 2 — T2: UPDATE ยอดเงิน Bob แล้ว COMMIT ทันที**

In [8]:
# UPDATE accounts SET balance = balance - 500 WHERE account_id = 2
conn_t2 = get_conn(autocommit=True) # command to COMMIT
run(conn_t2, "UPDATE accounts SET balance = balance - 500 WHERE account_id = 2;", fetch=False)
print("T2: หักเงิน Bob ไป 500 และ COMMIT แล้ว (autocommit=True)")
conn_t2.close()

T2: หักเงิน Bob ไป 500 และ COMMIT แล้ว (autocommit=True)


**ขั้นที่ 3 — T1: อ่านยอดเงิน Bob ครั้งที่ 2 (ใน transaction เดิม ที่ยังไม่ commit)**

In [9]:
cols, rows = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 2;")
print("T1 อ่านครั้งที่ 2 (READ COMMITTED):")
show(cols, rows)
run(conn_t1, "COMMIT;", fetch=False)
conn_t1.close()

T1 อ่านครั้งที่ 2 (READ COMMITTED):
balance
2500.00


**สังเกตผล A2 (READ COMMITTED):** ค่าที่อ่านได้ครั้งที่ 1 และครั้งที่ 2 **ต่างกัน** (2500.00) — นี่คือ Non-repeatable Read

ตอนนี้ให้ทำซ้ำขั้นตอนเดียวกันทั้งหมด แต่เปลี่ยน T1 เป็น `BEGIN ISOLATION LEVEL REPEATABLE READ;` แทน

In [10]:
# ทำซ้ำการทดลอง A2 ทั้งหมด แต่เปลี่ยน T1 เป็น ISOLATION LEVEL REPEATABLE READ
# โครงสร้างเหมือนเดิมทุกอย่าง เปลี่ยนแค่คำว่า READ COMMITTED -> REPEATABLE READ ในขั้นที่ 1
conn_t1 = get_conn()
run(conn_t1, "BEGIN ISOLATION LEVEL REPEATABLE READ;", fetch=False)
cols, rows = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 2;")
show(cols, rows)

conn_t2 = get_conn(autocommit=True)
run(conn_t2, "UPDATE accounts SET balance = balance - 500 WHERE account_id = 2;", fetch=False)
conn_t2.close()

cols, rows = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 2;")
show(cols, rows)
run(conn_t1, "COMMIT;", fetch=False)
conn_t1.close()

balance
2500.00
balance
2500.00


**คำถามชวนคิด A2:** ยอดเงิน Bob ในฐานข้อมูลตอนนี้คือเท่าไหร่ (หลังถูกหักไป 2 ครั้ง)? ลองรัน `SELECT` ตรวจสอบและอธิบายว่าทำไมค่าที่ T1 "เห็น" กับค่าจริงในฐานข้อมูลถึงต่างกันได้

In [11]:
check_conn = get_conn(autocommit=True)
cols, rows = run(check_conn, "SELECT * FROM accounts ORDER BY account_id;")
show(cols, rows)
check_conn.close()

account_id | owner_name | balance
1 | Alice | 5000.00
2 | Bob | 2000.00
3 | Charlie | 10000.00
4 | Diana | 2000.00


---
## Lab B — Lost Update

### B1. จำลองปัญหา Lost Update

**สถานการณ์:** Charlie มีเงิน 10,000 บาท มีสองรายการเกิดขึ้น "พร้อมกัน": ระบบตัดค่าธรรมเนียม 2,000 บาท (T1) และลูกค้าถอนเงิน 3,000 บาท ผ่านแอป (T2) — ทั้งสองอ่านยอดเงินตั้งต้นพร้อมกันก่อนที่ใครจะเขียนทับ

**ขั้นที่ 1 — T1 และ T2 อ่านยอดเงิน Charlie พร้อมกัน (ก่อนใครจะเขียนทับ)**

In [12]:
conn_t1 = get_conn()
conn_t2 = get_conn()
run(conn_t1, "BEGIN;", fetch=False)
run(conn_t2, "BEGIN;", fetch=False)

_, rows_t1 = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 3;")
_, rows_t2 = run(conn_t2, "SELECT balance FROM accounts WHERE account_id = 3;")
balance_t1 = float(rows_t1[0][0])
balance_t2 = float(rows_t2[0][0])
print(f"T1 อ่านได้: {balance_t1}   |   T2 อ่านได้: {balance_t2}")

T1 อ่านได้: 10000.0   |   T2 อ่านได้: 10000.0


**ขั้นที่ 2 — T1 หักค่าธรรมเนียม 2,000 บาท (จากค่าที่อ่านมา) แล้ว COMMIT**

In [13]:
# T1 คำนวณยอดใหม่ = balance_t1 - 2000 แล้ว UPDATE + COMMIT
new_balance_t1 = balance_t1 - 2000
run(conn_t1, "UPDATE accounts SET balance = %s WHERE account_id = 3;", (new_balance_t1,), fetch=False)
run(conn_t1, "COMMIT;", fetch=False)
print(f"T1: อัปเดตยอดเป็น {new_balance_t1} และ COMMIT แล้ว")

T1: อัปเดตยอดเป็น 8000.0 และ COMMIT แล้ว


**ขั้นที่ 3 — T2 ถอนเงิน 3,000 บาท (จากค่าที่อ่านมาตอนแรก ซึ่งยังไม่รู้ว่า T1 แก้ไปแล้ว) แล้ว COMMIT**

In [14]:
# T2 คำนวณยอดใหม่ = balance_t2 - 3000 แล้ว UPDATE + COMMIT
new_balance_t2 = balance_t2 - 3000
run(conn_t2, "UPDATE accounts SET balance = %s WHERE account_id = 3;", (new_balance_t2,), fetch=False)
run(conn_t2, "COMMIT;", fetch=False)
print(f"T2: อัปเดตยอดเป็น {new_balance_t2} และ COMMIT แล้ว")
conn_t1.close(); conn_t2.close()

T2: อัปเดตยอดเป็น 7000.0 และ COMMIT แล้ว


**ตรวจสอบผลลัพธ์**

In [15]:
check_conn = get_conn(autocommit=True)
cols, rows = run(check_conn, "SELECT balance FROM accounts WHERE account_id = 3;")
show(cols, rows)
check_conn.close()
print("\nยอดที่ถูกต้องควรเป็น 10000 - 2000 - 3000 = 5000")
print("แต่ยอดจริงในฐานข้อมูลคือเท่าไหร่? การหักของ T1 หายไปหรือไม่?")

balance
7000.00

ยอดที่ถูกต้องควรเป็น 10000 - 2000 - 3000 = 5000
แต่ยอดจริงในฐานข้อมูลคือเท่าไหร่? การหักของ T1 หายไปหรือไม่?


**สังเกตผล B1:** ยอดเงินสุดท้ายคือ **7000.00** (10000 - 3000) ไม่ใช่ 5000.00 ที่ควรจะเป็น — การหักค่าธรรมเนียม 2,000 ของ T1 **หายไป** เพราะ T2 เขียนทับด้วยค่าที่คำนวณจากยอดตั้งต้นเดิม (10000) โดยไม่รู้ว่า T1 แก้ไปแล้ว นี่คือ **Lost Update**

### B2. แก้ปัญหาด้วย `SELECT ... FOR UPDATE`

ก่อนอื่น รีเซ็ตยอดเงิน Charlie กลับเป็น 10,000

In [16]:
reset_conn = get_conn(autocommit=True)
run(reset_conn, "UPDATE accounts SET balance = 10000.00 WHERE account_id = 3;", fetch=False)
reset_conn.close()
print("รีเซ็ตยอดเงิน Charlie กลับเป็น 10000 แล้ว")

รีเซ็ตยอดเงิน Charlie กลับเป็น 10000 แล้ว


**ลำดับขั้นตอน:** ใช้ `SELECT ... FOR UPDATE` เพื่อล็อกแถวไว้ตั้งแต่ T1 เริ่มอ่าน — T2 จะต้อง **รอ** จนกว่า T1 จะ COMMIT

**ขั้นที่ 1 — T1: ล็อกแถวด้วย FOR UPDATE (ยังไม่ commit)**

In [17]:
conn_t1 = get_conn()
run(conn_t1, "BEGIN;", fetch=False)
conn_t1 = get_conn()
run(conn_t1, "BEGIN;", fetch=False)
_, rows_t1 = run(conn_t1, "SELECT balance FROM accounts WHERE account_id = 3 FOR UPDATE;")
balance_t1 = float(rows_t1[0][0])
print(f"T1: ล็อกแถวไว้แล้ว อ่านค่าได้ {balance_t1}")

T1: ล็อกแถวไว้แล้ว อ่านค่าได้ 10000.0


**ขั้นที่ 2 — T2: พยายาม FOR UPDATE แถวเดียวกัน (ตั้ง statement_timeout สั้นๆ เพื่อพิสูจน์ว่ามันถูกบล็อกจริง แทนที่จะรอเฉยๆ)**

In [18]:
conn_t2 = get_conn()
run(conn_t2, "SET statement_timeout = '3000';", fetch=False)  # รอสูงสุด 3 วินาที
run(conn_t2, "BEGIN;", fetch=False)
try:
    run(conn_t2, "SELECT balance FROM accounts WHERE account_id = 3 FOR UPDATE;")
except Exception as e:
    print("T2 ถูกบล็อก และ timeout ก่อนที่ T1 จะปล่อยล็อก:")
    print(type(e).__name__, "-", str(e).strip())
    conn_t2.rollback()


T2 ถูกบล็อก และ timeout ก่อนที่ T1 จะปล่อยล็อก:
QueryCanceled - canceling statement due to statement timeout
CONTEXT:  while locking tuple (0,10) in relation "accounts"


สังเกตผล: T2 ควรได้ error ว่า canceling statement due to statement timeout

**ขั้นที่ 3 — T1: COMMIT เพื่อปล่อยล็อก จากนั้น T2 ลองใหม่จะสำเร็จ**

In [19]:
# T1 UPDATE ยอดเงินลบ 2000 แล้ว COMMIT
run(conn_t1, "UPDATE accounts SET balance = %s WHERE account_id = 3;", (balance_t1 - 2000,), fetch=False)
run(conn_t1, "COMMIT;", fetch=False)
conn_t1.close()

In [20]:
conn_t2.rollback()
conn_t2.close()

verify_conn = get_conn(autocommit=True)
cols, rows = run(verify_conn, "SELECT balance FROM accounts WHERE account_id = 3;")
show(cols, rows)
verify_conn.close()
print("\nยอดเงินตอนนี้ควรเป็น 8000 (10000 - 2000) และไม่มีการ Lost Update เกิดขึ้นแล้ว")

balance
8000.00

ยอดเงินตอนนี้ควรเป็น 8000 (10000 - 2000) และไม่มีการ Lost Update เกิดขึ้นแล้ว


---
## Lab C — Deadlock

Lab นี้ต้องใช้ **Python threading** เพื่อให้ T1 และ T2 ทำงาน "พร้อมกันจริงๆ" (ต่างจาก Lab A/B ที่เราควบคุมลำดับเซลล์เอง) เพราะการเกิด Deadlock ต้องมีสอง transaction ค้างรอ lock กันและกัน **ในเวลาเดียวกัน**

**สถานการณ์:** T1 หักเงิน Alice (account 1) ก่อน แล้วค่อยไปแตะ Bob (account 2) — ส่วน T2 หักเงิน Bob (account 2) ก่อน แล้วค่อยไปแตะ Alice (account 1) — ลำดับการล็อกที่สวนทางกันนี้คือสูตรสำเร็จของ Deadlock

In [21]:
import threading
import time

def t1_job(result):
    conn = get_conn()
    try:
        run(conn, "BEGIN;", fetch=False)
        run(conn, "UPDATE accounts SET balance = balance - 100 WHERE account_id = 1;", fetch=False)
        print("T1: ล็อกบัญชี 1 (Alice) แล้ว")
        time.sleep(2)  # หน่วงเวลาให้ T2 ล็อกบัญชี 2 ได้ก่อน
        # T1 พยายาม UPDATE บัญชี 2 (Bob) ด้วย -- ควรถูกบล็อกและอาจเจอ deadlock
        run(conn, "UPDATE accounts SET balance = balance + 100 WHERE account_id = 2;", fetch=False)
        run(conn, "COMMIT;", fetch=False)
        result['t1'] = "COMMIT สำเร็จ"
    except Exception as e:
        conn.rollback()
        result['t1'] = f"ถูกยกเลิก: {type(e).__name__}: {str(e).strip()}"
    finally:
        conn.close()

def t2_job(result):
    conn = get_conn()
    try:
        run(conn, "BEGIN;", fetch=False)
        run(conn, "UPDATE accounts SET balance = balance - 50 WHERE account_id = 2;", fetch=False)
        print("T2: ล็อกบัญชี 2 (Bob) แล้ว")
        time.sleep(1)  # ให้แน่ใจว่า T1 ล็อกบัญชี 1 ไปก่อนแล้ว
        # T2 พยายาม UPDATE บัญชี 1 (Alice) ด้วย -- ควรถูกบล็อกและอาจเจอ deadlock
        run(conn, "UPDATE accounts SET balance = balance + 50 WHERE account_id = 1;", fetch=False)
        run(conn, "COMMIT;", fetch=False)
        result['t2'] = "COMMIT สำเร็จ"
    except Exception as e:
        conn.rollback()
        result['t2'] = f"ถูกยกเลิก: {type(e).__name__}: {str(e).strip()}"
    finally:
        conn.close()

result = {}
th1 = threading.Thread(target=t1_job, args=(result,))
th2 = threading.Thread(target=t2_job, args=(result,))
th1.start(); th2.start()
th1.join(); th2.join()

print("\n--- ผลลัพธ์ ---")
print("T1:", result.get('t1'))
print("T2:", result.get('t2'))


T1: ล็อกบัญชี 1 (Alice) แล้ว
T2: ล็อกบัญชี 2 (Bob) แล้ว

--- ผลลัพธ์ ---
T1: COMMIT สำเร็จ
T2: ถูกยกเลิก: DeadlockDetected: deadlock detected
DETAIL:  Process 4066 waits for ShareLock on transaction 20518; blocked by process 767.
Process 767 waits for ShareLock on transaction 20519; blocked by process 4066.
HINT:  See server log for query details.
CONTEXT:  while updating tuple (0,1) in relation "accounts"


ลองรันเซลล์ข้างบนซ้ำหลายๆ ครั้ง สังเกตว่า **T1 หรือ T2 ที่ถูกยกเลิกไม่จำเป็นต้องเป็นตัวเดิมเสมอ**

In [22]:
verify_conn = get_conn(autocommit=True)
cols, rows = run(verify_conn, "SELECT * FROM accounts ORDER BY account_id;")
show(cols, rows)
verify_conn.close()

account_id | owner_name | balance
1 | Alice | 4900.00
2 | Bob | 2100.00
3 | Charlie | 8000.00
4 | Diana | 2000.00
